# Transformer

In [42]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [43]:
# Cell 1: 데이터 로딩
import pandas as pd
import os

# 경로 설정
DATA_DIR = './data_filtering/filtered/'

# train 데이터 로드
train_data = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print("✅ train_data shape:", train_data.shape)
display(train_data.head())

# test 데이터 10개 로드
test_data_list = []
for i in range(10):
    test_path = os.path.join(DATA_DIR, f'TEST_{i:02d}.csv')
    df = pd.read_csv(test_path)
    test_data_list.append(df)
    print(f"✅ Loaded TEST_{i:02d}.csv | shape: {df.shape}")


✅ train_data shape: (102676, 6)


,date_ordinal,date,store_menu,store,menu,sales
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
3,738524,2023-01-04,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
4,738525,2023-01-05,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0


✅ Loaded TEST_00.csv | shape: (5404, 6)
✅ Loaded TEST_01.csv | shape: (5404, 6)
✅ Loaded TEST_02.csv | shape: (5404, 6)
✅ Loaded TEST_03.csv | shape: (5404, 6)
✅ Loaded TEST_04.csv | shape: (5404, 6)
✅ Loaded TEST_05.csv | shape: (5404, 6)
✅ Loaded TEST_06.csv | shape: (5404, 6)
✅ Loaded TEST_07.csv | shape: (5404, 6)
✅ Loaded TEST_08.csv | shape: (5404, 6)
✅ Loaded TEST_09.csv | shape: (5404, 6)


In [44]:
# Cell 2: 전처리 함수 정의 및 실행
import numpy as np

def preprocess_data(df):
    # 날짜형 변환
    df['date'] = pd.to_datetime(df['date'])
    df['date_ordinal'] = df['date'].map(pd.Timestamp.toordinal)

    # 'store_menu' 식별자 추가 (이미 있는 경우 생략 가능)
    if 'store_menu' not in df.columns:
        df['store_menu'] = df['store'] + "_" + df['menu']

    # 날짜 관련 파생 변수
    df['day_of_week'] = df['date'].dt.dayofweek      # 0=월 ~ 6=일
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day

    # 필요한 feature만 추출
    use_cols = [
        'date', 'date_ordinal', 'store_menu', 'sales',
        'day_of_week', 'is_weekend', 'month', 'day'
    ]
    return df[use_cols]

# train 데이터 전처리
train_data = preprocess_data(train_data)
display(train_data.head())

# test 데이터 전처리
for i in range(10):
    test_data_list[i] = preprocess_data(test_data_list[i])


,date,date_ordinal,store_menu,sales,day_of_week,is_weekend,month,day
0,2023-01-01,738521,느티나무 셀프BBQ_1인 수저세트,0,6,1,1,1
1,2023-01-02,738522,느티나무 셀프BBQ_1인 수저세트,0,0,0,1,2
2,2023-01-03,738523,느티나무 셀프BBQ_1인 수저세트,0,1,0,1,3
3,2023-01-04,738524,느티나무 셀프BBQ_1인 수저세트,0,2,0,1,4
4,2023-01-05,738525,느티나무 셀프BBQ_1인 수저세트,0,3,0,1,5


In [45]:
# Cell 3: Transformer 모델 정의
import torch
import torch.nn as nn

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout=0.1, output_len=7):
        super(TimeSeriesTransformer, self).__init__()
        
        self.model_type = 'Transformer'
        self.output_len = output_len
        self.d_model = d_model

        # 입력 임베딩: Linear -> Positional Encoding
        self.input_projection = nn.Linear(input_dim, d_model)

        # 포지셔널 인코딩
        self.positional_encoding = self._generate_positional_encoding(1000, d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 디코더 (예측용)
        self.decoder = nn.Linear(d_model, output_len)

    def forward(self, src):
        # src: [B, L, input_dim]
        src = self.input_projection(src)  # [B, L, d_model]

        pos_encoding = self.positional_encoding[:src.size(1), :].unsqueeze(0).to(src.device)  # [1, L, d_model]
        src = src + pos_encoding  # [B, L, d_model]

        src = src.permute(1, 0, 2)  # [L, B, d_model]
        output = self.transformer_encoder(src)  # [L, B, d_model]
        output = output.permute(1, 0, 2)  # [B, L, d_model]

        last_hidden = output[:, -1, :]  # [B, d_model]
        pred = self.decoder(last_hidden)  # [B, 7]
        return pred


    def _generate_positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe  # shape: [max_len, d_model] ← [1, max_len, d_model] ❌ X


In [49]:
from sklearn.model_selection import KFold
from torch.utils.data import Dataset, DataLoader, Subset
import torch
import pandas as pd
import numpy as np

# ✅ SequenceDataset 정의 (유지)
class SequenceDataset(Dataset):
    def __init__(self, df, input_len=28, output_len=7):
        self.input_len = input_len
        self.output_len = output_len
        self.data = []

        df = df.sort_values(['store_menu', 'date'])  # 시계열 정렬

        for sm in df['store_menu'].unique():
            df_sm = df[df['store_menu'] == sm]
            values = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values
            for i in range(max(0, len(values) - input_len - output_len + 1)):
                x = values[i:i+input_len]
                y = values[i+input_len:i+input_len+output_len, 0]
                self.data.append((x, y))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ✅ 1. 전체 시퀀스 생성
full_dataset = SequenceDataset(train_data)

# ✅ 2. 전체 시퀀스에 대해 KFold split
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(np.arange(len(full_dataset))))  # 시퀀스 인덱스를 기반으로 분할

# ✅ 3. Fold별로 DataLoader 생성
for fold, (train_idx, val_idx) in enumerate(folds):
    print(f"\n🌀 Fold {fold+1}")

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

    print(f"✅ Fold {fold+1} | Train samples: {len(train_subset)} | Val samples: {len(val_subset)}")

    # 🔁 모델 학습/검증 루프 연결 가능



🌀 Fold 1
✅ Fold 1 | Train samples: 76891 | Val samples: 19223

🌀 Fold 2
✅ Fold 2 | Train samples: 76891 | Val samples: 19223

🌀 Fold 3
✅ Fold 3 | Train samples: 76891 | Val samples: 19223

🌀 Fold 4
✅ Fold 4 | Train samples: 76891 | Val samples: 19223

🌀 Fold 5
✅ Fold 5 | Train samples: 76892 | Val samples: 19222


## OPTUNA 써서 튜닝 (더 수정해봐야됨)
- kfold 파라미터 설정
- optuna로 파라미터 최적화 후, k-fold validation 재수행해서 모델 학습하는 것도 고려

In [ ]:
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
import pandas as pd
import matplotlib.pyplot as plt
import os

# ✅ 직접 조정할 epoch 수
NUM_EPOCHS = 10
SAVE_DIR = './optuna_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ✅ Optuna objective 함수
def objective(trial):
    # 🔧 하이퍼파라미터 샘플링
    d_model = trial.suggest_categorical('d_model', [32, 64, 128])
    nhead = trial.suggest_categorical('nhead', [2, 4, 8])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    print(f"\n🧪 Trial {trial.number} Start")
    print(f"    ▶️ Params: d_model={d_model}, nhead={nhead}, num_layers={num_layers}, lr={lr:.5f}")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    folds = list(kf.split(np.arange(len(full_dataset))))
    fold_val_losses = []

    for fold, (train_idx, val_idx) in enumerate(folds):
        print(f"      🔄 Fold {fold+1}/3")

        train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=64, shuffle=True)
        val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=64, shuffle=False)

        model = TimeSeriesTransformer(
            input_dim=5,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            output_len=7
        ).to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                output = model(x_batch)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            print(f"        🏋️‍ Epoch {epoch+1}/{NUM_EPOCHS} - last batch loss: {loss.item():.4f}")

        # 🧪 Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                output = model(x_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * x_batch.size(0)
        val_loss /= len(val_loader.dataset)
        fold_val_losses.append(val_loss)
        print(f"        ✅ Fold {fold+1} Val Loss: {val_loss:.4f}")

    avg_val_loss = sum(fold_val_losses) / len(fold_val_losses)
    print(f"  📉 Trial {trial.number} Avg Val Loss: {avg_val_loss:.4f}")

    # ✅ 모델 저장 (best trial만 저장)
    if trial.study.best_trial is None or avg_val_loss < trial.study.best_trial.value:
        save_path = os.path.join(SAVE_DIR, f'best_model_trial_{trial.number}.pt')
        torch.save(model.state_dict(), save_path)
        trial.set_user_attr("best_model_path", save_path)
        print(f"  💾 Best model saved to: {save_path}")

    return avg_val_loss



In [48]:
# ✅ 튜닝 실행 및 결과 확인

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)  # trial 수 조절 가능

# ✅ 최적 결과 출력
print("🎯 Best hyperparameters:")
print(study.best_params)
print(f"📉 Best validation loss: {study.best_value:.4f}")
print(f"💾 Best model saved to: {study.best_trial.user_attrs['best_model_path']}")


[I 2025-08-07 14:44:45,247] A new study created in memory with name: no-name-954e9bc0-3f98-4027-bd84-338799061dd0
/home/wonjun/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



🧪 Trial 0 Start
    ▶️ Params: d_model=64, nhead=8, num_layers=2, lr=0.00021
      🔄 Fold 1/3
        🏋️‍ Epoch 1/10 - last batch loss: 30.5386
        🏋️‍ Epoch 2/10 - last batch loss: 461.2130
        🏋️‍ Epoch 3/10 - last batch loss: 39.5323
        🏋️‍ Epoch 4/10 - last batch loss: 8436.7070
        🏋️‍ Epoch 5/10 - last batch loss: 301.3192
        🏋️‍ Epoch 6/10 - last batch loss: 75.0950
        🏋️‍ Epoch 7/10 - last batch loss: 241.3076
        🏋️‍ Epoch 8/10 - last batch loss: 301.6588
        🏋️‍ Epoch 9/10 - last batch loss: 1166.8496
        🏋️‍ Epoch 10/10 - last batch loss: 192.3396
        ✅ Fold 1 Val Loss: 832.7722
      🔄 Fold 2/3
        🏋️‍ Epoch 1/10 - last batch loss: 1749.7560
        🏋️‍ Epoch 2/10 - last batch loss: 13.2542


[W 2025-08-07 14:47:50,422] Trial 0 failed with parameters: {'d_model': 64, 'nhead': 8, 'num_layers': 2, 'lr': 0.00021356810097586908} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_4624/2440889131.py", line 55, in objective
    loss.backward()
  File "/home/wonjun/.local/lib/python3.10/site-packages/torch/_tensor.py", line 521, in backward
    torch.autograd.backward(
  File "/home/wonjun/.local/lib/python3.10/site-packages/torch/autograd/__init__.py", line 289, in backward
    _engine_run_backward(
  File "/home/wonjun/.local/lib/python3.10/site-packages/torch/autograd/graph.py", line 768, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
KeyboardInterrupt
[W 2025-08-07 14:47:50,426] T

KeyboardInterrupt: 

In [ ]:
# 시각화
import optuna.visualization as vis

# ✅ 최적화 history
fig1 = vis.plot_optimization_history(study)
fig1.show()

# ✅ 파라미터 중요도
fig2 = vis.plot_param_importances(study)
fig2.show()


In [ ]:
# best 모델이 다음 경로에 저장되었을 때
# best_path = study.best_trial.user_attrs["best_model_path"]

# ✅ best 모델 로드
best_params = study.best_params
best_path = study.best_trial.user_attrs["best_model_path"]

best_model = TimeSeriesTransformer(
    input_dim=5,
    d_model=best_params['d_model'],
    nhead=best_params['nhead'],
    num_layers=best_params['num_layers'],
    output_len=7
).to(device)

best_model.load_state_dict(torch.load(best_path, map_location=device))
best_model.eval()
print(f"✅ Best model loaded from: {best_path}")


In [ ]:
def predict_test(test_df, model, device='cpu', input_len=28):
    preds = {}

    store_menus = test_df['store_menu'].unique()
    for sm in store_menus:
        df_sm = test_df[test_df['store_menu'] == sm].sort_values('date')
        x = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values[-input_len:]

        if x.shape[0] < input_len:
            continue  # skip 부족한 시계열

        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 28, 5]
        with torch.no_grad():
            output = model(x_tensor)  # [1, 7]
        preds[sm] = output.squeeze(0).cpu().numpy()
    
    return preds  # Dict[str, np.ndarray]


## OPTUNA 적용 안했을 때

In [50]:
# Cell 5: K-Fold 기반 모델 학습 루프
import torch.optim as optim
from tqdm import tqdm

# 하이퍼파라미터
input_dim = 5
d_model = 64
nhead = 4
num_layers = 2
output_len = 7
epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# K-Fold 학습 루프
for fold, (train_idx, val_idx) in enumerate(folds):
    print(f"\n🌀 Fold {fold+1}")

    # ✅ 데이터로더 정의
    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

    # ✅ 모델 초기화
    model = TimeSeriesTransformer(input_dim, d_model, nhead, num_layers, output_len=output_len).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # ✅ Epoch 학습
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        for x_batch, y_batch in tqdm(train_loader, desc=f"[Fold {fold+1}][Epoch {epoch}] Training", leave=False):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            output = model(x_batch)  # shape: [B, 7]
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x_batch.size(0)

        train_loss /= len(train_loader.dataset)

        # ✅ 검증
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                output = model(x_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * x_batch.size(0)

        val_loss /= len(val_loader.dataset)

        print(f"✅ [Fold {fold+1}] Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


/home/wonjun/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



🌀 Fold 1


✅ [Fold 1] Epoch 01 | Train Loss: 1325.4900 | Val Loss: 994.3499


✅ [Fold 1] Epoch 02 | Train Loss: 1033.3649 | Val Loss: 922.7057


✅ [Fold 1] Epoch 03 | Train Loss: 938.7675 | Val Loss: 836.7288


...

✅ [Fold 5] Epoch 16 | Train Loss: 797.3953 | Val Loss: 958.4202


✅ [Fold 5] Epoch 17 | Train Loss: 800.0891 | Val Loss: 850.9864


✅ [Fold 5] Epoch 18 | Train Loss: 770.5371 | Val Loss: 857.4930


✅ [Fold 5] Epoch 19 | Train Loss: 761.3662 | Val Loss: 964.6074


✅ [Fold 5] Epoch 20 | Train Loss: 777.2915 | Val Loss: 932.5542


In [ ]:
# Cell 5: 모델 학습 루프 정의
import torch.optim as optim
from tqdm import tqdm

# 하이퍼파라미터
input_dim = 5            # sales + day_of_week + is_weekend + month + day
d_model = 64
nhead = 4
num_layers = 2
output_len = 7
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 초기화
model = TimeSeriesTransformer(input_dim, d_model, nhead, num_layers, output_len=output_len).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 학습 루프
for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0
    for x_batch, y_batch in tqdm(train_loader, desc=f"[Epoch {epoch}] Training", leave=False):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        output = model(x_batch)  # shape: [B, 7]
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # 검증
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output = model(x_batch)
            loss = criterion(output, y_batch)
            val_loss += loss.item() * x_batch.size(0)

    val_loss /= len(val_loader.dataset)

    print(f"✅ Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [51]:
# Cell 6: 테스트 데이터 추론 함수
def predict_test_data(model, test_df, input_len=28, device='cpu'):
    model.eval()
    preds = {}
    
    store_menus = test_df['store_menu'].unique()

    for sm in store_menus:
        df_sm = test_df[test_df['store_menu'] == sm].sort_values('date')
        x = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values[-input_len:]
        
        if x.shape[0] != input_len:
            # 입력 길이가 부족할 경우 패스
            continue

        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 28, 5]
        with torch.no_grad():
            pred = model(x_tensor)  # [1, 7]
        preds[sm] = pred.squeeze(0).cpu().numpy()
    
    return preds  # Dict[str, np.ndarray]

# 모든 test 파일에 대해 예측 수행
all_test_preds = {}

for i in range(10):
    test_df = test_data_list[i]
    pred_dict = predict_test_data(model, test_df, device=device)
    all_test_preds[f'TEST_{i:02d}'] = pred_dict
    print(f"✅ Predicted TEST_{i:02d}: {len(pred_dict)} store_menus")


✅ Predicted TEST_00: 193 store_menus
✅ Predicted TEST_01: 193 store_menus
✅ Predicted TEST_02: 193 store_menus
✅ Predicted TEST_03: 193 store_menus
✅ Predicted TEST_04: 193 store_menus
✅ Predicted TEST_05: 193 store_menus
✅ Predicted TEST_06: 193 store_menus
✅ Predicted TEST_07: 193 store_menus
✅ Predicted TEST_08: 193 store_menus
✅ Predicted TEST_09: 193 store_menus


In [52]:
all_test_preds

{'TEST_00': {'느티나무 셀프BBQ_1인 수저세트': array([-1.0204648 , -0.37515885,  0.16060668, -0.22457361,  0.27302533,
         -0.4107737 ,  0.04977936], dtype=float32),
  '느티나무 셀프BBQ_BBQ55(단체)': array([20.870152, 20.110361, 19.998049, 19.04972 , 19.645882, 19.150757,
         20.164434], dtype=float32),
  '느티나무 셀프BBQ_대여료 30,000원': array([-8.455584 , -7.336044 , -6.5827546, -6.769932 , -6.3105426,
         -7.0553327, -6.7918077], dtype=float32),
  '느티나무 셀프BBQ_대여료 60,000원': array([-7.543733 , -6.483632 , -5.757388 , -5.9691834, -5.5038257,
         -6.2406325, -5.9520717], dtype=float32),
  '느티나무 셀프BBQ_대여료 90,000원': array([-9.142003 , -7.982677 , -7.210659 , -7.3789153, -6.920592 ,
         -7.6690116, -7.4235024], dtype=float32),
  '느티나무 셀프BBQ_본삼겹 (단품,실내)': array([-9.094004 , -7.937676 , -7.167034 , -7.336601 , -6.878034 ,
         -7.6260896, -7.379311 ], dtype=float32),
  '느티나무 셀프BBQ_스프라이트 (단체)': array([8.940188, 8.94409 , 9.185221, 8.542486, 9.08998 , 8.492845,
         9.207226], dtype=float

In [53]:
# Cell 7: 음수 예측값 → 0으로 클리핑
for test_key in all_test_preds:
    for sm_key in all_test_preds[test_key]:
        all_test_preds[test_key][sm_key] = np.clip(all_test_preds[test_key][sm_key], a_min=0, a_max=None)

print("✅ 모든 음수 예측값을 0으로 변환 완료")


✅ 모든 음수 예측값을 0으로 변환 완료


In [54]:
# Cell 8: 예측 결과를 sample_submission에 채워넣기
import pandas as pd

# sample submission 불러오기
submission = pd.read_csv('./result/sample_submission.csv', index_col=0)
print("✅ 불러온 submission shape:", submission.shape)

# 예측값 채워넣기
for test_key in all_test_preds:  # e.g., TEST_00
    for store_menu in all_test_preds[test_key]:
        preds = all_test_preds[test_key][store_menu]  # shape: (7,)
        for day_offset in range(7):
            row_idx = f"{test_key}+{day_offset+1}일"
            if store_menu in submission.columns:
                submission.at[row_idx, store_menu] = preds[day_offset]

# 최종 확인
display(submission.head(10))


✅ 불러온 submission shape: (70, 193)


/tmp/ipykernel_4624/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.16060668230056763' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykernel_4624/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '20.87015151977539' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykernel_4624/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '8.94018840789795' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykern

,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,느티나무 셀프BBQ_쌈장,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
영업일자,,,,,,,,,,,,,,,,,,,,,
TEST_00+1일,0.000000,20.870152,0.000000,0,0,0,8.940188,0,0.0,0,...,0.0,8.284308,2.367847,0.0,61.882580,45.211735,0.0,49.231205,0.0,27.623112
TEST_00+2일,0.000000,20.110361,0.000000,0,0,0,8.944090,0,0.0,0,...,0.0,8.335820,2.797428,0.0,58.433685,42.879612,0.0,46.626789,0.0,26.446363
TEST_00+3일,0.160607,19.998049,0.000000,0,0,0,9.185221,0,0.0,0,...,0.0,8.597861,3.233990,0.0,57.079975,42.038933,0.0,45.661964,0.0,26.138569
TEST_00+4일,0.000000,19.049721,0.000000,0,0,0,8.542486,0,0.0,0,...,0.0,7.971931,2.760900,0.0,55.116119,40.482059,0.0,44.006474,0.0,25.018282
TEST_00+5일,0.273025,19.645882,0.000000,0,0,0,9.089980,0,0.0,0,...,0.0,8.513222,3.274446,0.0,55.866146,41.161488,0.0,44.706909,0.0,25.626772
TEST_00+6일,0.000000,19.150757,0.000000,0,0,0,8.492845,0,0.0,0,...,0.0,7.907482,2.618536,0.0,55.740303,40.874840,0.0,44.462078,0.0,25.180643
TEST_00+7일,0.049779,20.164434,0.000000,0,0,0,9.207226,0,0.0,0,...,0.0,8.603294,3.165078,0.0,57.740639,42.481274,0.0,46.166378,0.0,26.357582
TEST_01+1일,0.000000,2.237307,0.400859,0,0,0,0.000000,0,0.0,0,...,0.0,0.000000,3.332541,0.0,17.807777,22.521784,0.0,22.098129,0.0,10.761628
TEST_01+2일,0.000000,2.665145,0.951440,0,0,0,0.000000,0,0.0,0,...,0.0,0.000000,3.691890,0.0,17.255926,21.681601,0.0,21.283386,0.0,10.656974


In [56]:
# 저장
submission.to_csv('./result/Vanilla_transformer_kfold.csv')
print("✅ 최종 제출 파일 저장 완료: ./result/Vanilla_transformer_kfold.csv")


✅ 최종 제출 파일 저장 완료: ./result/Vanilla_transformer_kfold.csv
